# PyIceberg Read Path Deep Dive

> **Companion notebook for the Read Path slides** | Iceberg Summit, April 8, 2026

This notebook walks through every stage of the Iceberg read path with live code you can run and inspect. Each section maps to a slide in the deck.

**Pre-requisites:**
- Docker containers running (`docker-compose up -d`)
- Data loaded: run the write-path workshop first (creates 3 snapshots, 150 rows)

---

| # | Section | Slide | What You'll See |
|---|---------|-------|-----------------|
| 1 | Connect to catalog | Title | Entry point -- catalog lookup |
| 2 | Metadata hierarchy | Slide 2 | Walk the 4-layer tree in code |
| 3 | Full scan (no filter) | Slide 4 | plan_files() baseline |
| 4 | Filtered scan -- pruning in action | Slide 3 | Two-level pruning proof |
| 5 | Pruning funnel -- stats deep dive | Slide 5 | Column min/max elimination |
| 6 | Time travel | Slide 6 | Read historical snapshots |
| 7 | Snapshot isolation | Slide 6 | Concurrent read safety |
| 8 | Column projection | Slide 4 | Read only needed columns |
| 9 | Schema evolution on read | Slide 7 | Field IDs, type widening, NULLs |
| 10 | Partition evolution | Slide 7 | Hidden partitioning |
| 11 | Q&A prep -- inspect tables | Slide 9 | inspect.snapshots/manifests/files |

In [ ]:
#!pip install -r requirements.txt

---
## ⚙️ Prerequisites & Workshop Setup

### What's Running (Docker Stack)

Start the stack before running this notebook:

```bash
cd ../infra && docker-compose up -d
```

| Service | URL | Purpose |
|---------|-----|---------|
| REST Catalog | http://localhost:8181 | Iceberg catalog — tracks table locations |
| MinIO S3 | http://localhost:9000 | Object storage — Parquet + metadata files |
| MinIO Console | http://localhost:9001 | Browse S3 files visually in browser |
| PostgreSQL | localhost:5432 | Catalog backend — stores table pointers |

> **Important:** Run the **write-path workshop** first to create the table with 3 snapshots and 150 rows.

In [ ]:
# ─── Infrastructure Health Check ─────────────────────────────────────────────────────────────────
# Run this first! Confirms all Docker services are reachable before we start.

import urllib.request
import urllib.error

checks = [
    ("REST Catalog",  "http://localhost:8181/v1/config"),
    ("MinIO S3",      "http://localhost:9000/minio/health/live"),
    ("MinIO Console", "http://localhost:9001"),
]

print("Checking workshop infrastructure...\n")
all_ok = True

for name, url in checks:
    try:
        urllib.request.urlopen(url, timeout=5)
        print(f"  ✅ {name:20s} {url}")
    except Exception as e:
        print(f"  ❌ {name:20s} {url}  -- {e}")
        all_ok = False

if all_ok:
    print("\n✅ All services healthy -- ready to go!")
else:
    print("\n❌ Some services are down. Run: cd ../infra && docker-compose up -d")

---
## 1. Connect to REST Catalog

**What happens here:** Your code asks the REST catalog "where is this table?" The catalog is backed by PostgreSQL -- it stores the pointer to the current `metadata.json`. This is the very first step in every read path.

**Maps to:** Slide 0 (Title) + Slide 2 (Metadata Hierarchy, Layer 1)

In [ ]:
from pyiceberg.catalog.rest import RestCatalog
import datetime

catalog = RestCatalog(
    name="workshop",
    **{
        "uri": "http://localhost:8181",
        "s3.endpoint": "http://localhost:9000",
        "s3.access-key-id": "admin",
        "s3.secret-access-key": "password",
        "s3.path-style-access": "true",
        "py-io-impl": "pyiceberg.io.pyarrow.PyArrowFileIO",
    },
)

table = catalog.load_table("pyiceberg_demo.sensor_readings")

print(f"Table loaded:  {table.name()}")
print(f"Location:      {table.location()}")
print(f"Format ver:    {table.format_version}")
print(f"Snapshots:     {len(table.history())}")
print(f"Schema fields: {[f.name for f in table.schema().fields]}")

**Expected output:**
```
Table loaded:  sensor_readings
Location:      s3://warehouse/pyiceberg_demo/sensor_readings
Format ver:    2
Snapshots:     3
Schema fields: ['sensor_id', 'ts', 'co2_ppm', 'ch4_ppb', 'batch_id']
```

**Sequence diagram — what happens under the hood:**

```mermaid
sequenceDiagram
    autonumber
    participant Code as Your Code
    participant RC as RestCatalog<br/><br/>catalog/rest/__init__.py
    participant SRV as REST Catalog Server<br/><br/>localhost:8181
    participant PG as PostgreSQL<br/><br/>Metadata Store
    participant S3 as MinIO / S3

    Code->>RC: catalog.load_table("pyiceberg_demo.sensor_readings")
    RC->>SRV: GET /v1/namespaces/pyiceberg_demo/tables/sensor_readings
    SRV->>PG: SELECT metadata_location FROM iceberg_tables
    PG-->>SRV: s3://warehouse/.../metadata/00003-<uuid>.metadata.json
    SRV->>S3: GET metadata.json
    S3-->>SRV: full TableMetadata JSON
    SRV-->>RC: TableMetadata (schema, specs, snapshots, current-snapshot-id)
    RC-->>Code: Table object (metadata in memory, no data read yet)

    Note over Code,S3: The Table object now holds the FULL metadata tree in memory — no data files touched yet
```

**What just happened:**
- `catalog.load_table()` sends `GET /v1/namespaces/pyiceberg_demo/tables/sensor_readings` to the REST server
- The server reads the table's `metadata_location` from PostgreSQL, fetches the `metadata.json` from S3, and returns the full `TableMetadata`
- The client now has **everything** in memory: schema, partition specs, all snapshots, sort orders — but **zero data files** have been read
- This is the entry point for every read operation: scan, time travel, schema inspection

---
## 2. Walk the Metadata Hierarchy

**What we're doing:** Walking the 4-layer metadata tree that Iceberg uses to go from "which table?" to "which Parquet files?". This is the core of the read path — each layer narrows down what to read.

```
catalog → metadata.json → manifest list → manifest files → data files
 (1 call)    (in memory)     (1 Avro read)   (N Avro reads)   (only matching files)
```

| Layer | File | What It Contains | Pruning Gate |
|-------|------|------------------|--------------|
| 1 | `metadata.json` | Schema, partition spec, snapshot list | — (already in memory) |
| 2 | `snap-*.avro` | List of manifest files for this snapshot | — |
| 3 | `*-m0.avro` | Per-file partition bounds + column stats | **Manifest pruning + metrics pruning** |
| 4 | `*.parquet` | Actual data (columnar) | Only files that survived pruning |

In [ ]:
# ── Layer 1: metadata.json (lives inside TableMetadata, already loaded) ──

metadata = table.metadata
print("=== Layer 1: metadata.json ===")
print(f"  format-version:     {metadata.format_version}")
print(f"  table-uuid:         {metadata.table_uuid}")
print(f"  location:           {metadata.location}")
print(f"  schemas:            {len(metadata.schemas)} version(s)")
print(f"  partition-specs:    {len(metadata.partition_specs)}")
print(f"  snapshots:          {len(metadata.snapshots)}")
print(f"  current-snapshot-id: {metadata.current_snapshot_id}")
print(f"  properties:         {dict(metadata.properties)}")

In [ ]:
# ── Layer 2: Manifest List (one per snapshot -- points to manifests) ──

snap = table.current_snapshot()
print("=== Layer 2: Manifest List ===")
print(f"  snapshot_id:    {snap.snapshot_id}")
print(f"  timestamp:      {datetime.datetime.fromtimestamp(snap.timestamp_ms / 1000)}")
print(f"  operation:      {snap.summary.operation.value}")
print(f"  manifest list:  {snap.manifest_list.split('/')[-1]}")
print(f"  total-records:  {snap.summary.get('total-records', 'N/A')}")

io = table.io
manifests = snap.manifests(io)
print(f"\n  This manifest list contains {len(manifests)} manifest file(s):")

In [ ]:
# ── Layer 3: Manifest Files (file index with column stats) ──
# Each manifest lists data files and their per-column statistics.
# This is PRUNING GATE #2 -- column min/max lets us skip files.

print("=== Layer 3: Manifest Files ===\n")
for i, mf in enumerate(manifests):
    print(f"  Manifest [{i}]: {mf.manifest_path.split('/')[-1]}")
    print(f"    added_files:    {mf.added_files_count}")
    print(f"    existing_files: {mf.existing_files_count}")
    print(f"    deleted_files:  {mf.deleted_files_count}")

    # Partition field summaries -- this is what manifest pruning checks
    if mf.partitions:
        for ps in mf.partitions:
            print(f"    partition summary: contains_null={ps.contains_null}, "
                  f"lower={ps.lower_bound}, upper={ps.upper_bound}")
    print()

In [ ]:
# ── Layer 4: Data Files (the actual Parquet files) ──
# Iceberg NEVER reads these during planning -- only after deciding a file is relevant.

print("=== Layer 4: Data Files ===\n")
total_files = 0
for i, mf in enumerate(manifests):
    entries = mf.fetch_manifest_entry(io, discard_deleted=True)
    for e in entries:
        df = e.data_file
        total_files += 1
        print(f"  [{i}] {df.file_path.split('/')[-1]}")
        print(f"      record_count:  {df.record_count}")
        print(f"      file_size:     {df.file_size_in_bytes:,} bytes")
        print(f"      lower_bounds:  {dict(df.lower_bounds) if df.lower_bounds else 'N/A'}")
        print(f"      upper_bounds:  {dict(df.upper_bounds) if df.upper_bounds else 'N/A'}")
        print(f"      null_counts:   {dict(df.null_value_counts) if df.null_value_counts else 'N/A'}")
        print()

print(f"--- S3 Call Summary ---")
print(f"  1 metadata.json  (from catalog)")
print(f"  1 manifest list  (snap-*.avro)")
print(f"  {len(manifests)} manifest file(s)  (*-m0.avro)")
print(f"  {total_files} data file(s)     (*.parquet)")
print(f"  Total S3 reads for planning: ~{2 + len(manifests)}  (data reads are separate)")

**Expected output:**
```
=== Layer 1: metadata.json ===
  format-version:     2
  schemas:            1 version(s)
  partition-specs:    1
  snapshots:          3
  ...

=== Layer 2: Manifest List ===
  snapshot_id:    <id>
  manifest files: 3
  ...

=== Layer 3: Manifest Files ===
  Manifest [0]: <uuid>-m0.avro
    added_files: 1 ...
  ...

=== Layer 4: Data Files ===
  [0] <uuid>.parquet  record_count: 50  ...
  Total data files: 3
```

> **Check MinIO:** Open [localhost:9001](http://localhost:9001) → `warehouse/pyiceberg_demo/sensor_readings/` to see these files on disk.

**Sequence diagram — how the read path walks the metadata tree:**

```mermaid
sequenceDiagram
    autonumber
    participant Code as Your Code
    participant TM as TableMetadata<br/><br/>(in memory)
    participant S3 as MinIO / S3

    Code->>TM: table.current_snapshot()
    TM-->>Code: Snapshot (snapshot_id, manifest_list path)

    Code->>S3: snap.manifests(io) — read manifest list Avro
    S3-->>Code: [ManifestFile, ManifestFile, ManifestFile]

    loop For each ManifestFile
        Code->>S3: mf.fetch_manifest_entry(io) — read manifest Avro
        S3-->>Code: [ManifestEntry] with DataFile stats
    end

    Note over Code,S3: Now we know every file's path, record count, column bounds, and partition values — without opening any Parquet
```

**What just happened:**
- Layer 1 (`metadata.json`): Already in memory from `load_table()`. Contains schema, partition spec, all snapshots.
- Layer 2 (`snap-*.avro`): `snap.manifests(io)` reads the Avro manifest list from S3/MinIO.
- Layer 3 (`*-m0.avro`): Each `ManifestFile` has `partition_field_summary` (pruning gate #1) and per-file column stats (pruning gate #2).
- Layer 4 (`*.parquet`): `fetch_manifest_entry()` gives us `DataFile` objects with `lower_bounds`, `upper_bounds`, `null_value_counts`.
- Source: `pyiceberg/manifest.py` -- `read_manifest_list()`, `ManifestFile.fetch_manifest_entry()`

---
## 3. Full Table Scan (no filter -- baseline)

**What happens:** `plan_files()` with no filter returns ALL data files. This is our baseline to compare against filtered scans.

**Maps to:** Slide 4 (Scan Planning Internals)

---
## 3. Full Scan (No Filter)

**What we're doing:** Running `table.scan()` with no filter and no column selection. This is the baseline — every file survives planning. We'll compare this against filtered scans in the next section to see pruning in action.

In [ ]:
# Full scan -- no filter, no column selection
# table.scan() is LAZY -- nothing runs until plan_files() or to_arrow()

scan = table.scan()

# plan_files() is the scan planning phase:
#   1. Resolve snapshot
#   2. Read manifest list
#   3. For each manifest: manifest_evaluator (no filter = keep all)
#   4. For each file: partition_evaluator + metrics_evaluator (no filter = keep all)
#   5. Return FileScanTask per surviving file

all_tasks = list(scan.plan_files())
print(f"plan_files() returned {len(all_tasks)} FileScanTask(s)\n")

for i, task in enumerate(all_tasks):
    print(f"  Task [{i}]: {task.file.file_path.split('/')[-1]}")
    print(f"    record_count: {task.file.record_count}")
    print(f"    file_size:    {task.file.file_size_in_bytes:,} bytes")

# Now actually read the data
full_df = scan.to_arrow()
print(f"\nTotal rows returned: {len(full_df)}")
print(f"Columns: {full_df.column_names}")

**Expected output:**
```
plan_files() returned 3 file scan tasks (no filter = all files)

Task 0: <uuid>.parquet  (50 rows, ~X KB)
Task 1: <uuid>.parquet  (50 rows, ~X KB)
Task 2: <uuid>.parquet  (50 rows, ~X KB)

Full table: 150 rows, 5 columns
```

**What just happened:**
- `table.scan()` creates a `DataScan` object -- purely lazy, no I/O yet.
- `plan_files()` calls `_plan_files_local()` which reads the manifest list, builds evaluators, and iterates manifests.
- With no filter, all evaluators return `True` (keep everything).
- Source: `pyiceberg/table/__init__.py` -- `DataScan.plan_files()`, `_plan_files_local()`, `scan_plan_helper()`

---
## 4. Filtered Scan -- Two-Level Pruning in Action

**This is the O(1) magic.** Watch how Iceberg eliminates files without opening them:
1. **Manifest pruning** (coarse): skip entire manifests based on partition summaries
2. **Metrics pruning** (fine): skip individual files based on column min/max stats

**Maps to:** Slide 3 (Two-Level Pruning)

---
## 4. Filtered Scan — Two-Level Pruning in Action

**What we're doing:** Applying row filters to see Iceberg's O(1) file elimination. The scan planner checks each file's recorded column statistics (`lower_bounds`, `upper_bounds`) against your predicate — and skips files that **cannot** contain matching rows.

| Pruning Level | Where | What It Checks |
|---------------|-------|----------------|
| Level 1: Manifest pruning | `_ManifestEvalVisitor` | Partition field summaries per manifest |
| Level 2: Metrics pruning | `_InclusiveMetricsEvaluator` | Per-file column min/max/null stats |

In [ ]:
from pyiceberg.expressions import EqualTo, GreaterThan, And

# ── Filter 1: Equality predicate on batch_id ──
# Each append wrote a different batch_id (1, 2, 3), so each file has
# non-overlapping batch_id ranges. Iceberg should skip 2 of 3 files.

filtered_scan = table.scan(row_filter=EqualTo("batch_id", 2))
filtered_tasks = list(filtered_scan.plan_files())

print(f"=== Filter: batch_id == 2 ===")
print(f"  Unfiltered: {len(all_tasks)} file scan tasks")
print(f"  Filtered:   {len(filtered_tasks)} file scan task(s)")
print(f"  Files eliminated: {len(all_tasks) - len(filtered_tasks)}\n")

for task in filtered_tasks:
    print(f"  Surviving file: {task.file.file_path.split('/')[-1]}")
    print(f"    record_count: {task.file.record_count}")

filtered_df = filtered_scan.to_arrow()
print(f"\n  Rows returned: {len(filtered_df)}")
print(f"  All batch_id == 2? {all(v == 2 for v in filtered_df['batch_id'].to_pylist())}")

**Expected output:**
```
=== Filter: batch_id == 2 ===
  Unfiltered: 3 file scan tasks
  Filtered:   1 file scan task(s)
  Files eliminated: 2 of 3
```

> **Why it works:** Each batch wrote a different `batch_id` (1, 2, 3) into separate Parquet files. The `_InclusiveMetricsEvaluator` checks `upper_bounds[batch_id]` and `lower_bounds[batch_id]` for each file and immediately eliminates files where `batch_id == 2` is impossible.

In [ ]:
# ── Filter 2: Range predicate on co2_ppm ──
# This uses the InclusiveMetricsEvaluator -- checks upper_bounds/lower_bounds
# to decide if a file MIGHT contain matching rows.

range_scan = table.scan(row_filter=GreaterThan("co2_ppm", 405.0))
range_tasks = list(range_scan.plan_files())

print(f"=== Filter: co2_ppm > 405 ===")
print(f"  Unfiltered: {len(all_tasks)} file scan tasks")
print(f"  Filtered:   {len(range_tasks)} file scan task(s)")
print(f"  Files eliminated: {len(all_tasks) - len(range_tasks)}\n")

# Show WHY each file was kept or would be skipped
print("  Why? Let's check each file's co2_ppm bounds:")
for i, mf in enumerate(manifests):
    entries = mf.fetch_manifest_entry(io, discard_deleted=True)
    for e in entries:
        df = e.data_file
        ub = df.upper_bounds
        lb = df.lower_bounds
        # Find the field ID for co2_ppm
        co2_field = table.schema().find_field("co2_ppm")
        co2_id = co2_field.field_id
        upper = ub.get(co2_id, b"?") if ub else "N/A"
        lower = lb.get(co2_id, b"?") if lb else "N/A"
        print(f"    {df.file_path.split('/')[-1]}:")
        print(f"      co2_ppm bounds: lower={lower}, upper={upper}")
        print(f"      -> {'KEEP (upper > 405)' if upper != 'N/A' else 'unknown'}")

range_df = range_scan.to_arrow()
print(f"\n  Rows matching co2_ppm > 405: {len(range_df)}")

**Expected output:**
```
=== Filter: co2_ppm > 405 ===
  Unfiltered: 3 file scan tasks
  Filtered:   2-3 file scan task(s)
  Files eliminated: 0-1 of 3
```

> **Note:** CO2 values are random ±5 around 400 ppm, so some files' `upper_bounds[co2_ppm]` may be below 405 — those get pruned. Your results may vary by random seed.

**Sequence diagram — how filtered scan planning works:**

```mermaid
sequenceDiagram
    autonumber
    participant Code as Your Code
    participant DS as DataScan<br/><br/>table/__init__.py
    participant ML as ManifestList<br/><br/>snap-*.avro
    participant ME as _ManifestEvalVisitor<br/><br/>Partition pruning
    participant IM as _InclusiveMetricsEvaluator<br/><br/>Column stats pruning
    participant S3 as MinIO / S3

    Code->>DS: table.scan(row_filter=EqualTo("batch_id", 2))
    DS->>DS: Bind filter to schema, resolve snapshot

    DS->>S3: Read manifest list (snap-*.avro)
    S3-->>DS: 3 ManifestFile entries

    loop For each ManifestFile
        DS->>ME: manifest_evaluator(partition_filter)
        ME-->>DS: keep / skip manifest

        alt Manifest kept
            DS->>S3: Read manifest entries
            S3-->>DS: [ManifestEntry with DataFile stats]
            loop For each DataFile
                DS->>IM: metrics_evaluator(lower_bounds, upper_bounds)
                IM-->>DS: keep / skip file
            end
        end
    end

    DS-->>Code: [FileScanTask] — only surviving files
```

**What just happened:**
- `EqualTo("batch_id", 2)` creates a `BooleanExpression` that gets bound to the schema.
- `manifest_evaluator()` checks partition field summaries -- can skip entire manifests.
- `_InclusiveMetricsEvaluator` checks per-file `lower_bounds` / `upper_bounds` -- skips files where the value **cannot** exist.
- The evaluator is **conservative** (inclusive): keeps files that *might* match. False positives OK, false negatives would lose data.
- Source: `pyiceberg/expressions/visitors.py` -- `manifest_evaluator()`, `_InclusiveMetricsEvaluator`

---
## 5. Pruning Funnel -- Stats Deep Dive

**Maps to:** Slide 5 (Pruning Funnel)

Let's build a complete picture of every file's stats and see exactly which files survive each predicate.

---
## 5. Pruning Funnel — Stats Deep Dive

**What we're doing:** Building a stats table from all data files to see exactly what the evaluator sees, then comparing how many files survive different predicates.

In [ ]:
import pyarrow as pa

# Build a stats table from all data files -- shows what the evaluator sees
schema_fields = table.schema()
stats_rows = []

for mf in manifests:
    entries = mf.fetch_manifest_entry(io, discard_deleted=True)
    for e in entries:
        df = e.data_file
        row = {
            "file": df.file_path.split("/")[-1][:30],
            "records": df.record_count,
            "size_kb": round(df.file_size_in_bytes / 1024, 1),
        }
        # Extract bounds for each column
        for field in schema_fields.fields:
            fid = field.field_id
            lb = df.lower_bounds.get(fid, None) if df.lower_bounds else None
            ub = df.upper_bounds.get(fid, None) if df.upper_bounds else None
            nc = df.null_value_counts.get(fid, None) if df.null_value_counts else None
            row[f"{field.name}_lower"] = str(lb) if lb else "N/A"
            row[f"{field.name}_upper"] = str(ub) if ub else "N/A"
            row[f"{field.name}_nulls"] = nc if nc is not None else "N/A"
        stats_rows.append(row)

stats_table = pa.table({k: [r.get(k) for r in stats_rows] for k in stats_rows[0]})
print("=== Data File Statistics (what the pruning evaluators see) ===\n")
print(stats_table.to_pandas().to_string(index=False))

In [ ]:
# ── Pruning funnel comparison ──
# Compare how many files survive different predicates

from pyiceberg.expressions import (
    EqualTo, NotEqualTo, GreaterThan, GreaterThanOrEqual,
    LessThan, LessThanOrEqual, IsNull, NotNull, In
)

test_filters = [
    ("No filter (baseline)",         None),
    ("batch_id == 1",                EqualTo("batch_id", 1)),
    ("batch_id == 2",                EqualTo("batch_id", 2)),
    ("batch_id == 3",                EqualTo("batch_id", 3)),
    ("co2_ppm > 405",                GreaterThan("co2_ppm", 405.0)),
    ("co2_ppm < 395",                LessThan("co2_ppm", 395.0)),
    ("batch_id == 2 AND co2_ppm>405", And(EqualTo("batch_id", 2), GreaterThan("co2_ppm", 405.0))),
]

print(f"{'Filter':<35} {'Files':>5} {'Rows':>6}  Pruning")
print("-" * 70)

for label, filt in test_filters:
    scan_obj = table.scan(row_filter=filt) if filt else table.scan()
    tasks = list(scan_obj.plan_files())
    df_result = scan_obj.to_arrow()
    eliminated = len(all_tasks) - len(tasks)
    pct = f"({eliminated}/{len(all_tasks)} files skipped)" if filt else "(baseline)"
    print(f"  {label:<33} {len(tasks):>5} {len(df_result):>6}  {pct}")

**Key takeaway:** Iceberg checked each file's column bounds and eliminated files without opening them. At scale (thousands of files), this turns minutes of S3 LIST + file scanning into milliseconds of metadata reads.

---
## 6. Time Travel -- Read Historical Snapshots

**What happens:** Every `append()` created a new snapshot. We can scan any old snapshot by ID. The read path is identical -- just starts from a different snapshot's manifest list.

**Maps to:** Slide 6 (Time Travel & Snapshot Isolation)

---
## 6. Time Travel — Read Historical Snapshots

**What we're doing:** Every `append()` created a new snapshot. We can scan any old snapshot by ID. The read path is identical — just starts from a different snapshot's manifest list.

> **Use cases:** Reproducible ML training, auditing ("what did the table look like last Tuesday?"), debugging data quality issues.

In [ ]:
# ── Snapshot history ──

history = table.history()
print("=== Snapshot History ===\n")
for h in history:
    ts = datetime.datetime.fromtimestamp(h.timestamp_ms / 1000)
    is_current = " (current)" if h.snapshot_id == snap.snapshot_id else ""
    print(f"  snapshot {h.snapshot_id}  @  {ts}{is_current}")

# ── Time travel: read after first batch only ──

first_snapshot_id = history[0].snapshot_id
old_scan = table.scan(snapshot_id=first_snapshot_id)
old_tasks = list(old_scan.plan_files())
old_df = old_scan.to_arrow()

print(f"\n=== Time Travel ===")
print(f"  Current table:     {len(full_df)} rows  ({len(all_tasks)} files)")
print(f"  After first batch: {len(old_df)} rows  ({len(old_tasks)} file(s))")
print(f"\n  Same table, different point in time!")

# ── Compare all snapshots ──
print(f"\n=== Row Count at Each Snapshot ===\n")
for h in history:
    ts = datetime.datetime.fromtimestamp(h.timestamp_ms / 1000)
    snap_df = table.scan(snapshot_id=h.snapshot_id).to_arrow()
    print(f"  snapshot {h.snapshot_id}  @  {ts}  ->  {len(snap_df)} rows")

**Expected output:**
```
=== Snapshot History ===

  snapshot <id1>  @  2026-04-06 ...  (current)
  snapshot <id2>  @  2026-04-06 ...
  snapshot <id3>  @  2026-04-06 ...

=== Time Travel: First Snapshot Only ===
  Files: 1 scan task(s)
  Rows:  50 (just batch 1)
```

**Sequence diagram — time travel read path:**

```mermaid
sequenceDiagram
    autonumber
    participant Code as Your Code
    participant DS as DataScan
    participant TM as TableMetadata<br/><br/>(in memory)
    participant S3 as MinIO / S3

    Code->>DS: table.scan(snapshot_id=first_snapshot_id)
    DS->>TM: Look up snapshot by ID in metadata.snapshots[]
    TM-->>DS: Snapshot 1 (manifest_list = snap-<id1>.avro)

    Note over DS,S3: From here, identical to a normal scan — just uses snapshot 1's manifest list

    DS->>S3: Read snap-<id1>.avro
    S3-->>DS: 1 ManifestFile (only batch 1's data)
    DS->>S3: Read manifest entries
    S3-->>DS: 1 DataFile (50 rows)
    DS-->>Code: [FileScanTask] → to_arrow() → 50 rows
```

**What just happened:**
- `table.scan(snapshot_id=X)` resolves the specific snapshot from `metadata.snapshots[]` instead of using `refs["main"]`.
- The rest of the read path is identical -- just operating on that snapshot's manifest list.
- Old Parquet files, manifests, snapshots are **never modified** -- immutability is the key.

**Use cases:**
- "Sensor was miscalibrated -- show pre-correction data"
- Reproducible ML training -- pin to a specific data version
- Auditing -- "what did the table look like when that report ran?"

---
## 7. Snapshot Isolation

**No locks. Readers never block writers. Writers never block readers.**

Your scan is pinned to a snapshot. Even if new snapshots are committed during your query, you read from the one you started with.

---
## 7. Snapshot Isolation

**What we're doing:** Demonstrating that a pinned scan always returns the same result, even if new data is written concurrently. This is Iceberg's MVCC (multi-version concurrency control) — readers never block writers, writers never block readers.

In [ ]:
# Demonstrate snapshot isolation:
# 1. Start a scan (pin to current snapshot)
# 2. Simulate a "concurrent write" by noting what we'd see
# 3. Show that our scan result doesn't change

current_snap_id = table.current_snapshot().snapshot_id
print(f"=== Snapshot Isolation Demo ===\n")
print(f"  1. Pinned scan to snapshot: {current_snap_id}")

# This scan is pinned -- even if someone writes new data right now,
# we'll still see exactly this snapshot's data.
pinned_scan = table.scan(snapshot_id=current_snap_id)
pinned_df = pinned_scan.to_arrow()
print(f"  2. Rows in pinned scan: {len(pinned_df)}")

# Compare with "latest" scan
latest_df = table.scan().to_arrow()
print(f"  3. Rows in latest scan: {len(latest_df)}")

print(f"\n  Both return {len(pinned_df)} rows because no concurrent writes happened.")
print(f"  But if another process appended data between steps 1 and 3,")
print(f"  the pinned scan would STILL return {len(pinned_df)} rows.")
print(f"  The latest scan would see the new data.")
print(f"\n  This works because everything is immutable files in object storage.")

**Expected output:**
```
=== Snapshot Isolation Demo ===

  1. Pinned scan to snapshot: <id>
  2. Rows from pinned scan: 150
  3. Even if another writer appends data now, this scan still returns 150 rows
```

---
## 8. Column Projection -- Read Only What You Need

**Parquet is columnar** -- Iceberg can skip columns you don't ask for. This matters when tables have 100+ columns but you only need 3.

**Maps to:** Slide 4 (Scan Planning -- schema projection step)

In [ ]:
# ── Full table (all columns) ──
all_cols_df = table.scan().to_arrow()
print(f"=== Column Projection ===\n")
print(f"  All columns:  {all_cols_df.column_names}")
print(f"  Table size:   {all_cols_df.nbytes:,} bytes\n")

# ── Narrow scan (only 2 columns) ──
narrow_scan = table.scan(selected_fields=("sensor_id", "co2_ppm"))
narrow_df = narrow_scan.to_arrow()
print(f"  Selected:     ('sensor_id', 'co2_ppm')")
print(f"  Got columns:  {narrow_df.column_names}")
print(f"  Table size:   {narrow_df.nbytes:,} bytes")
print(f"  Savings:      {100 * (1 - narrow_df.nbytes / all_cols_df.nbytes):.0f}% less data read")

# ── You can also combine projection with filtering ──
combo_scan = table.scan(
    row_filter=EqualTo("batch_id", 2),
    selected_fields=("sensor_id", "co2_ppm", "batch_id"),
)
combo_df = combo_scan.to_arrow()
print(f"\n  Combined (batch_id==2, 3 cols): {len(combo_df)} rows, {combo_df.column_names}")

**Expected output:**
```
=== Column Projection ===

  All columns:  ['sensor_id', 'ts', 'co2_ppm', 'ch4_ppb', 'batch_id']
  Table size:   ~X bytes

  Selected:     ('sensor_id', 'co2_ppm')
  Got columns:  ['sensor_id', 'co2_ppm']
  Table size:   ~Y bytes  (smaller!)
  Savings:      ~Z%
```

**What just happened:**
- `selected_fields` creates a **projected schema** -- only these columns are read from Parquet.
- `prune_columns(file_schema, projected_field_ids)` tells PyArrow to skip unneeded column chunks entirely.
- On wide tables (50-100+ columns), this is a massive I/O savings.
- Source: `pyiceberg/io/pyarrow.py` -- column pruning in `_task_to_record_batches()`

---
## 9. Schema Evolution on Read

**Iceberg uses field IDs, not column positions or names.** This means:
- Renaming a column doesn't break anything
- Adding a column fills NULLs for old files (no rewrite)
- Type widening (int32 -> int64) happens at read time

**Maps to:** Slide 7 (Schema & Partition Evolution)

In [ ]:
# ── Show current schema with field IDs ──
print("=== Schema Evolution ===\n")
print("Current schema (notice field IDs are permanent, not positions):\n")
for field in table.schema().fields:
    print(f"  field_id={field.field_id:<3}  name={field.name:<20}  type={field.field_type}")

# ── Show all historical schemas ──
print(f"\nSchema versions: {len(table.schemas())}")
for i, schema in enumerate(table.schemas()):
    field_names = [f.name for f in schema.fields]
    print(f"  Schema {i} (id={schema.schema_id}): {field_names}")

In [ ]:
# ── Add a new column to demonstrate schema evolution ──
# This adds 'humidity' to the schema. Old Parquet files don't have it,
# but reads will return NULL for that column -- no rewrite needed.

from pyiceberg.types import FloatType

try:
    with table.update_schema() as update:
        update.add_column("humidity", FloatType(), doc="Relative humidity %")
    print("Added 'humidity' column to schema")
except Exception as e:
    print(f"Column may already exist: {e}")

# Reload table to get updated schema
table = catalog.load_table("workshop.sensor_readings")

print(f"\nUpdated schema:\n")
for field in table.schema().fields:
    print(f"  field_id={field.field_id:<3}  name={field.name:<20}  type={field.field_type}")

# ── Read with evolved schema ──
# Old files don't have 'humidity' -- Iceberg fills NULL
evolved_df = table.scan().to_arrow()
print(f"\nRows: {len(evolved_df)}")
print(f"Columns: {evolved_df.column_names}")
print(f"\nHumidity column (all NULL for old files):")
print(f"  null count: {evolved_df['humidity'].null_count}")
print(f"  sample:     {evolved_df['humidity'][:5].to_pylist()}")

**Expected output:**
```
=== Schema Evolution ===

Current schema (notice field IDs are permanent, not positions):

  field_id=1    name=sensor_id           type=string
  field_id=2    name=ts                  type=timestamp
  field_id=3    name=co2_ppm             type=double
  field_id=4    name=ch4_ppb             type=double
  field_id=5    name=batch_id            type=long

Added 'humidity' column to schema
  field_id=6    name=humidity            type=float

Reading old data with new schema: humidity = NULL for all pre-existing rows (no rewrite!)
```

**What just happened:**
- `update_schema()` creates a new schema version with a fresh field ID for the new column.
- `ArrowProjectionVisitor` handles: type casting, column reordering, missing columns filled with NULL arrays, timestamp normalization.
- `lower_bounds` / `upper_bounds` use **field IDs** (numbers), not column names -- this is why rename doesn't break stats.
- Source: `pyiceberg/io/pyarrow.py` -- `_to_requested_schema()`, `ArrowProjectionVisitor`

---
## 10. Partition Evolution

**Iceberg lets you change partitioning without rewriting data.** New writes use the new partition spec; old files keep their old partitions. Iceberg handles both during query planning.

**Maps to:** Slide 7 (Partition Evolution + Hidden Partitioning)

In [ ]:
# ── Current partition spec ──
print("=== Partition Evolution ===\n")
print(f"Current partition spec (id={table.spec().spec_id}):")
if table.spec().fields:
    for pf in table.spec().fields:
        print(f"  field_id={pf.source_id}, name={pf.name}, transform={pf.transform}")
else:
    print("  (unpartitioned)")

print(f"\nAll historical partition specs: {len(table.specs())}")
for spec_id, spec in table.specs().items():
    fields = [f"{f.name}({f.transform})" for f in spec.fields] if spec.fields else ["(none)"]
    print(f"  Spec {spec_id}: {fields}")

# ── Hidden partitioning concept ──
print(f"""
=== Hidden Partitioning ===

  Hive way (must know partition structure):
    SELECT * FROM readings WHERE year=2024 AND month=3 AND day=15

  Iceberg way (natural filter, Iceberg applies transform internally):
    SELECT * FROM readings WHERE timestamp >= '2024-03-15'

  Users don't need to know partition columns -- Iceberg handles it.
  Each manifest tracks its partition_spec_id, so mixed specs work.""")

**Expected output:**
```
=== Partition Evolution ===

Current partition spec (id=0):
  (unpartitioned)

All historical partition specs: 1
```

> **Key insight:** The partition spec lives in `metadata.json`, not in the Parquet files. Iceberg can evolve partitioning without rewriting data — old files keep their original spec, new writes use the new spec.

---
## 11. Q&A Prep -- Inspect Tables

**Maps to:** Slide 9 (Conference Q&A Quick Reference)

The `table.inspect` API returns Arrow tables with full metadata -- great for debugging and conference demos.

In [ ]:
# ── inspect.snapshots() -- one row per snapshot ──
print("=== inspect.snapshots() ===\n")
snapshots_df = table.inspect.snapshots().to_pandas()
print(snapshots_df.to_string(index=False))
print(f"\n  {len(snapshots_df)} snapshot(s) total")

In [ ]:
# ── inspect.manifests() -- one row per manifest file ──
print("=== inspect.manifests() ===\n")
manifests_df = table.inspect.manifests().to_pandas()
# Show key columns
cols = [c for c in manifests_df.columns if c in [
    'path', 'length', 'partition_spec_id', 'added_snapshot_id',
    'added_data_files_count', 'existing_data_files_count', 'deleted_data_files_count',
    'added_rows_count', 'existing_rows_count',
]]
print(manifests_df[cols].to_string(index=False) if cols else manifests_df.to_string(index=False))
print(f"\n  {len(manifests_df)} manifest file(s) total")

In [ ]:
# ── inspect.files() -- one row per data file with full stats ──
print("=== inspect.files() ===\n")
files_df = table.inspect.files().to_pandas()
# Show key columns
cols = [c for c in files_df.columns if c in [
    'file_path', 'file_format', 'record_count', 'file_size_in_bytes',
    'column_sizes', 'value_counts', 'null_value_counts',
    'lower_bounds', 'upper_bounds',
]]
if cols:
    display_df = files_df[cols].copy()
    # Shorten file paths for display
    if 'file_path' in display_df.columns:
        display_df['file_path'] = display_df['file_path'].apply(lambda x: x.split('/')[-1][:30] if isinstance(x, str) else x)
    print(display_df.to_string(index=False))
else:
    print(files_df.to_string(index=False))
print(f"\n  {len(files_df)} data file(s) total")

**Expected output:**
```
=== inspect.snapshots() ===
  3 snapshot(s) total with IDs, timestamps, operations

=== inspect.manifests() ===
  Manifest files with added/existing/deleted counts per snapshot

=== inspect.files() ===
  Data files with record counts, sizes, column bounds
```

> **Tip:** These return `pa.Table` / `pd.DataFrame` objects — you can filter, sort, and export them for debugging.

**Under the hood:**
- `table.inspect` returns `InspectTable` which provides `.snapshots()`, `.manifests()`, `.files()`, `.history()`, `.refs()` etc.
- These all return `pa.Table` objects -- you can filter, sort, and explore them.
- Source: `pyiceberg/table/inspect.py`

---
## Summary

```
Read Path Flow:

  table.scan(row_filter, selected_fields, snapshot_id)
      |
      v
  Snapshot Resolution     -->  Which version of the table?
      |
      v
  Schema Projection       -->  Which columns to read?
      |
      v
  Manifest Pruning        -->  Skip entire manifest files (partition summaries)
      |
      v
  Partition Pruning       -->  Skip files by exact partition value
      |
      v
  Metrics Pruning         -->  Skip files by column min/max stats
      |
      v
  Residual Computation    -->  What filter remains for row-level?
      |
      v
  Parallel Parquet Read   -->  Read only surviving files, only needed columns
      |
      v
  Schema Evolution Proj   -->  Type cast, fill NULLs, reorder columns
      |
      v
  pa.Table / DataFrame    -->  Final result
```

**Key numbers:**
- Planning: ~5 S3 reads (constant, regardless of table size)
- Data: only matching files, only needed columns
- No locks, no dirty reads, full snapshot isolation